In [2]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder
# 제출 파일 생성 관련
import os
import zipfile

# 데이터 처리 및 분석
import pandas as pd
import numpy as np
from scipy import stats
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt

# 머신러닝 전처리
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

# 머신러닝 모델
import xgboost as xgb

# 합성 데이터 생성
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer

# To ignore all warnings
import warnings

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score


# ✅ 데이터 불러오기 및 전처리
train_all = pd.read_csv("data/train_kor.csv")
test_all = pd.read_csv("data/test_kor.csv")
x_id = test_all["샘플 식별자 번호"]
train_all.rename(columns={"사기 시나리오 (예측 목표)": "label"}, inplace=True)

drop_cols = ['고객명', '주민번호', '거래에 사용한 단말기 IP주소', '거래에 사용한 단말기 MAC주소',
             '암호화된 계좌번호', '수취인 계좌번호', '샘플 식별자 번호']
train_all.drop(columns=drop_cols, inplace=True, errors='ignore')
test_all.drop(columns=drop_cols, inplace=True, errors='ignore')

In [3]:
train_x = train_all 
test_x = test_all

In [4]:
## 파생변수 추가 코드
# 1. 최근 잔고 급감 여부 (초기 잔고의 70% 이상 하락)
train_x['Is_Sudden_Balance_Drop'] = (train_x['거래 후 잔액'] < train_x['거래 전 잔액'] * 0.3).astype(int)
test_x['Is_Sudden_Balance_Drop'] = (test_x['거래 후 잔액'] < test_x['거래 전 잔액'] * 0.3).astype(int)

# 2. 고액 이체 탐지 (거래금액 / 하루한도 비율)
train_x['Amount_Per_Limit_Ratio'] = train_x['이체 금액'] / (train_x['1일 거래 한도'] + 1)
test_x['Amount_Per_Limit_Ratio'] = test_x['이체 금액'] / (test_x['1일 거래 한도'] + 1)

# 5. (VPN , 루팅 , 로밍) 중 하나이면서, 고객 등급이 C인지 여부
train_x['B_Flag_GradeC_DeviceAnomaly'] = (
    (train_x['고객 등급'] == 'C') &
    ((train_x['모바일 로밍 여부'] == 1) | (train_x['탈옥 및 루팅 여부'] == 1) | (train_x['VPN 사용 여부'] == 1))).astype(int)

test_x['B_Flag_GradeC_DeviceAnomaly'] = (
    (test_x['고객 등급'] == 'C') &
    ((test_x['모바일 로밍 여부'] == 1) |(test_x['탈옥 및 루팅 여부'] == 1) | (test_x['VPN 사용 여부'] == 1))).astype(int)

# 6. 유휴 계좌의 갑작스러운 거래 시도
# train_x['Inactive_Account_Suddenly_Used'] = ((train_x['7일 거래내역 중 미거래 계좌 여부'] == 1) & (train_x['이체 금액'] > 0)).astype(int)
# test_x['Inactive_Account_Suddenly_Used'] = ((test_x['7일 거래내역 중 미거래 계좌 여부'] == 1) & (test_x['이체 금액'] > 0)).astype(int)

# # 7. 접속 실패 후 성공 거래 (3회 이상 실패 시 플래그)
# train_x['High_Connection_Failure_Flag'] = (train_x['거래 시스템 접속 실패 횟수'] >= 3).astype(int)
# test_x['High_Connection_Failure_Flag'] = (test_x['거래 시스템 접속 실패 횟수'] >= 3).astype(int)

# 8. 거래금액 / 최근 한달 최대 거래금액
train_x['Amount_vs_Monthly_Max'] = train_x['이체 금액'] / (train_x['1개월 거래내역 중 최대 이체(출금) 금액'] + 1)
test_x['Amount_vs_Monthly_Max'] = test_x['이체 금액'] / (test_x['1개월 거래내역 중 최대 이체(출금) 금액'] + 1)

# # 11. 동일 수취계좌로 반복 전송 여부 (같은 계좌로 3회 초과 송금)
# transfer_counts_train = train_x.groupby(['암호화된 계좌번호', '수취인 계좌번호']).size().rename('이체횟수')
# train_x = train_x.merge(transfer_counts_train, on=['암호화된 계좌번호', '수취인 계좌번호'], how='left')
# train_x['Same_Account_Repeated_Transfer'] = (train_x['이체횟수'] > 3).astype(int)

# transfer_counts_test = test_x.groupby(['암호화된 계좌번호', '수취인 계좌번호']).size().rename('이체횟수')
# test_x = test_x.merge(transfer_counts_test, on=['암호화된 계좌번호', '수취인 계좌번호'], how='left')
# test_x['Same_Account_Repeated_Transfer'] = (test_x['이체횟수'] > 3).astype(int)

# 12. 고객 나이가 60세 이상이면서, 대출 유형이 담보대출인지 여부
# 기준 연도 설정
current_year = 2024

train_x['고객 나이'] = current_year - train_x['고객 출생년도']
train_x['L_Is_Elderly_Secured_Loan'] = (
    (train_x['고객 나이'] >= 60) & (train_x['대출 신청 유형(a: 없음, b: 신용대출, c: 담보대출, d: 할부금융, e: 기타)'] == 'c')
    ).astype(int)

test_x['고객 나이'] = current_year - test_x['고객 출생년도']
test_x['L_Is_Elderly_Secured_Loan'] = (
    (test_x['고객 나이'] >= 60) & (test_x['대출 신청 유형(a: 없음, b: 신용대출, c: 담보대출, d: 할부금융, e: 기타)'] == 'c')
    ).astype(int)

# A. 거리
train_x['Is_Large_Distance'] = (train_x['직전 거래 발생지와의 거리 차이'] >= 300).astype(int)
train_x['A_VPN_and_Large_Distance'] = (
    ((train_x['거래 시스템 접근 매체(a: ID/PW 로그인, b: 패턴, c: 생체로그인, d: 금융/공동 인증서, e: 사설인증서, f: 보안카드, g: OTP, h: 보안카드+OTP)'] == 'a') | (train_x['거래 시스템 접근 매체(a: ID/PW 로그인, b: 패턴, c: 생체로그인, d: 금융/공동 인증서, e: 사설인증서, f: 보안카드, g: OTP, h: 보안카드+OTP)'] == 'b'))  & (train_x['Is_Large_Distance'] == 1)
    ).astype(int)

test_x['Is_Large_Distance'] = (test_x['직전 거래 발생지와의 거리 차이'] >= 300).astype(int)
test_x['A_VPN_and_Large_Distance'] = (
    ((test_x['거래 시스템 접근 매체(a: ID/PW 로그인, b: 패턴, c: 생체로그인, d: 금융/공동 인증서, e: 사설인증서, f: 보안카드, g: OTP, h: 보안카드+OTP)'] == 'a') | (test_x['거래 시스템 접근 매체(a: ID/PW 로그인, b: 패턴, c: 생체로그인, d: 금융/공동 인증서, e: 사설인증서, f: 보안카드, g: OTP, h: 보안카드+OTP)'] == 'b')) & (test_x['Is_Large_Distance'] == 1)
    ).astype(int)
# B. 고객 등급이 ‘C’이면서, (VPN , 루팅 , 로밍) 중 하나가 사용되었을 경우
train_x['B_Flag_GradeC_DeviceAnomaly'] = (
    (train_x['고객 등급'] == 'C') &
    ((train_x['모바일 로밍 여부'] == 1) | (train_x['탈옥 및 루팅 여부'] == 1) | (train_x['VPN 사용 여부'] == 1))).astype(int)

test_x['B_Flag_GradeC_DeviceAnomaly'] = (
    (test_x['고객 등급'] == 'C') &
    ((test_x['모바일 로밍 여부'] == 1) |(test_x['탈옥 및 루팅 여부'] == 1) | (test_x['VPN 사용 여부'] == 1))).astype(int)

# C. 키로깅을 통한 정보 탈취 거래
train_x['C_Keylogging_Risk'] = ((train_x['거래 시스템 접속 실패 횟수'] >= 3) & (train_x['키로깅 여부'] == 1)).astype(int)
test_x['C_Keylogging_Risk'] = ((test_x['거래 시스템 접속 실패 횟수'] >= 3) & (test_x['키로깅 여부'] == 1)).astype(int)
# D. 7일 거래내역 중 미사용 단말 여부가 1이고, 거래 시스템 접근 매체가 ID /PW 로그인 및 패턴, 거래 성공/실패 여부이 0이고, 에러코드가 0인 경우
train_x['D_Scripted_Like_Transaction'] = (
    (train_x['7일 거래내역 중 미사용 단말 여부'] == 1) &
    (train_x['거래 시스템 접근 매체(a: ID/PW 로그인, b: 패턴, c: 생체로그인, d: 금융/공동 인증서, e: 사설인증서, f: 보안카드, g: OTP, h: 보안카드+OTP)'].isin(['a', 'b'])) &
    (train_x['거래 성공/실패 여부'] == 0) &
    (train_x['에러코드(a: 에러없음, b: 시스템 오류, c: 잔액부족, d: 이체한도초과, e: 계좌정보 오류, f: 계좌이체 거부)'] == 'a')
    ).astype(int)

test_x['D_Scripted_Like_Transaction'] = (
    (test_x['7일 거래내역 중 미사용 단말 여부'] == 1) &
    (test_x['거래 시스템 접근 매체(a: ID/PW 로그인, b: 패턴, c: 생체로그인, d: 금융/공동 인증서, e: 사설인증서, f: 보안카드, g: OTP, h: 보안카드+OTP)'].isin(['a', 'b'])) &
    (test_x['거래 성공/실패 여부'] == 0) &
    (test_x['에러코드(a: 에러없음, b: 시스템 오류, c: 잔액부족, d: 이체한도초과, e: 계좌정보 오류, f: 계좌이체 거부)'] == 'a')
    ).astype(int)
# E. 보이스피싱 인출 시도
train_x['E_Vishing_Withdrawal_Risk'] = ((train_x['거래 채널'] == 'ATM') & (train_x['7일 거래내역 중 1천만원 이상 입금 여부'] == 1)).astype(int)
test_x['E_Vishing_Withdrawal_Risk'] = ((test_x['거래 채널'] == 'ATM') & (test_x['7일 거래내역 중 1천만원 이상 입금 여부'] == 1)).astype(int)

# F. 고액 거래
train_x['G_MoneyLaundering_Risk'] = ((train_x['7일 거래내역 중 1천만원 이상 입금 여부'] == 1) & (train_x['30일 이내 계좌 정지 해제 여부'] == 1) & train_x['거래에 사용한 단말기 OS'] == 'Others').astype(int)
test_x['G_MoneyLaundering_Risk'] = ((test_x['7일 거래내역 중 1천만원 이상 입금 여부'] == 1) & (test_x['30일 이내 계좌 정지 해제 여부'] == 1) & test_x['거래에 사용한 단말기 OS'] == 'Others').astype(int)

train_x['F_MoneyLaundering_Risk'] = (
    (train_x['거래 채널'] == 'Others')
    & (train_x['거래에 사용한 단말기 OS'].isin(['Others', 'Windows']))
    ).astype(int)

test_x['F_MoneyLaundering_Risk'] = (
    (test_x['거래 채널'] == 'Others')
    & (test_x['거래에 사용한 단말기 OS'].isin(['Others', 'Windows']))
    ).astype(int)
# G. 자금세탁 거래
train_x['G_MoneyLaundering_Risk'] = ((train_x['7일 거래내역 중 1천만원 이상 입금 여부'] == 1) & (train_x['30일 이내 계좌 정지 해제 여부'] == 1)).astype(int)
test_x['G_MoneyLaundering_Risk'] = ((test_x['7일 거래내역 중 1천만원 이상 입금 여부'] == 1) & (test_x['30일 이내 계좌 정지 해제 여부'] == 1)).astype(int)

# # H. 소액 분산 이체
# same_amount_counts_train = train_x.groupby(['암호화된 계좌번호', '이체 금액']).size().rename('동일금액이체횟수').reset_index()
# train_x = train_x.merge(same_amount_counts_train, on=['암호화된 계좌번호', '이체 금액'], how='left')
# train_x['H_Repeated_Same_Amount_Transfer'] = (train_x['동일금액이체횟수'] >= 3).astype(int)

# same_amount_counts_test = test_x.groupby(['암호화된 계좌번호', '이체 금액']).size().rename('동일금액이체횟수').reset_index()
# test_x = test_x.merge(same_amount_counts_test, on=['암호화된 계좌번호', '이체 금액'], how='left')
# test_x['H_Repeated_Same_Amount_Transfer'] = (test_x['동일금액이체횟수'] >= 3).astype(int)

# J. 대포통장 거래
train_x['J_Suspicious_Small_Repeated_Transfer'] = ((train_x['수취계좌의 거래중지계좌 해당 여부'] == 1) & (train_x['7일 거래내역 중 미거래 계좌 여부'] == 1)).astype(int)
test_x['J_Suspicious_Small_Repeated_Transfer'] = ((test_x['수취계좌의 거래중지계좌 해당 여부'] == 1) & (test_x['7일 거래내역 중 미거래 계좌 여부'] == 1)).astype(int)

# K. 부업 사기 및 공범 계좌 활용
train_x['K_Affiliate_Scam_Risk'] = ((train_x['3시간 이내 해당 수취계좌에 이체 횟수'] >= 2) & (train_x['해당 수취계좌와 거래한 횟수'] >= 2)).astype(int)
test_x['K_Affiliate_Scam_Risk'] = ((test_x['3시간 이내 해당 수취계좌에 이체 횟수'] >= 2) & (test_x['해당 수취계좌와 거래한 횟수'] >= 2)).astype(int)


In [5]:
def add_custom_features(df):
    df = df.copy()
    security_cols = [
        '탈옥 및 루팅 여부',
        '모바일 로밍 여부',
        'VPN 사용 여부'
    ]
    security_flag_cnt = df[security_cols].sum(axis=1)
    df['모바일 보안_2plus'] = (security_flag_cnt >= 2).astype(int)
    return df


add_custom_features(train_x)
add_custom_features(test_x)

,Unnamed: 0,고객 출생년도,고객 성별,고객 등록일자,고객 등급,3개월 이내 금융/공동인증서 발급 여부,3개월 이내 사설인증서 발급 여부,3개월 이내 보안카드 및 OTP 발급 여부,3개월 이내 개인정보 수정 여부,탈옥 및 루팅 여부,...,Is_Large_Distance,A_VPN_and_Large_Distance,C_Keylogging_Risk,D_Scripted_Like_Transaction,E_Vishing_Withdrawal_Risk,G_MoneyLaundering_Risk,F_MoneyLaundering_Risk,J_Suspicious_Small_Repeated_Transfer,K_Affiliate_Scam_Risk,모바일 보안_2plus
0,0,1960,female,2003-01-07 10:59:08,E,1,0,0,0,1,...,0,0,0,1,0,0,0,1,0,0
1,1,1960,female,2003-01-07 10:59:08,E,1,1,1,1,0,...,0,0,0,0,0,0,1,0,0,0
2,2,1951,male,2003-01-06 18:10:55,B,1,1,1,1,0,...,0,0,0,0,0,1,0,0,1,0
3,3,1999,female,2003-01-08 05:28:53,B,0,1,1,1,0,...,0,0,0,1,0,0,0,1,0,0
4,4,1996,female,2003-01-17 03:37:22,A,0,1,0,0,1,...,0,0,0,1,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,119995,2004,male,2024-12-24 02:29:11,D,1,0,0,0,0,...,1,1,0,1,0,0,1,0,0,0
119996,119996,1955,male,2010-07-15 01:27:01,B,0,0,1,0,0,...,0,0,0,1,0,0,1,0,0,0
119997,119997,1987,female,2011-09-30 02:32:19,B,0,1,1,1,1,...,0,0,0,1,1,1,0,0,0,0
119998,119998,2004,female,2024-04-02 11:15:06,C,1,1,1,1,0,...,1,1,0,1,0,0,0,1,0,0


In [6]:
def add_custom_features_1(df):
    df = df.copy()


    # 터미널 악의적 행동 플래그
    terminal_cols = [
        '전화번호 조작 여부',
        '원격제어 여부',
        '템퍼링 여부',
        '피싱 여부',
        '신뢰할 수 없는 인증서 사용 여부',
        '키로깅 여부'
    ]
    terminal_flag_cnt = df[terminal_cols].sum(axis=1)
    df['terminal_malicious_2plus'] = (terminal_flag_cnt >= 2).astype(int)
    return df

add_custom_features_1(train_x)
add_custom_features_1(test_x)

,Unnamed: 0,고객 출생년도,고객 성별,고객 등록일자,고객 등급,3개월 이내 금융/공동인증서 발급 여부,3개월 이내 사설인증서 발급 여부,3개월 이내 보안카드 및 OTP 발급 여부,3개월 이내 개인정보 수정 여부,탈옥 및 루팅 여부,...,Is_Large_Distance,A_VPN_and_Large_Distance,C_Keylogging_Risk,D_Scripted_Like_Transaction,E_Vishing_Withdrawal_Risk,G_MoneyLaundering_Risk,F_MoneyLaundering_Risk,J_Suspicious_Small_Repeated_Transfer,K_Affiliate_Scam_Risk,terminal_malicious_2plus
0,0,1960,female,2003-01-07 10:59:08,E,1,0,0,0,1,...,0,0,0,1,0,0,0,1,0,1
1,1,1960,female,2003-01-07 10:59:08,E,1,1,1,1,0,...,0,0,0,0,0,0,1,0,0,0
2,2,1951,male,2003-01-06 18:10:55,B,1,1,1,1,0,...,0,0,0,0,0,1,0,0,1,0
3,3,1999,female,2003-01-08 05:28:53,B,0,1,1,1,0,...,0,0,0,1,0,0,0,1,0,0
4,4,1996,female,2003-01-17 03:37:22,A,0,1,0,0,1,...,0,0,0,1,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,119995,2004,male,2024-12-24 02:29:11,D,1,0,0,0,0,...,1,1,0,1,0,0,1,0,0,0
119996,119996,1955,male,2010-07-15 01:27:01,B,0,0,1,0,0,...,0,0,0,1,0,0,1,0,0,0
119997,119997,1987,female,2011-09-30 02:32:19,B,0,1,1,1,1,...,0,0,0,1,1,1,0,0,0,0
119998,119998,2004,female,2024-04-02 11:15:06,C,1,1,1,1,0,...,1,1,0,1,0,0,0,1,0,0


In [7]:
def add_custom_features_2(df):
    df = df.copy()

    # 출금 후 남은 잔액이 기존 금액의 10% 이하
    df['거래금액'] = df['거래 전 잔액'] - df['거래 후 잔액']

    df['잔액 변화 비율'] = df.apply(
        lambda row: row['거래금액'] / row['거래 전 잔액']
        if row['거래 전 잔액'] > 0 else 0,
        axis=1
    )

    # 변화 비율이 90% 이상이면, 잔액이 10% 이하로 줄어든 것
    df['잔액이 크게 감소'] = (df['잔액 변화 비율'] >= 0.9).astype(int)
    df.drop(columns = '잔액 변화 비율')

    return df

add_custom_features_2(train_x)
add_custom_features_2(test_x)

,Unnamed: 0,고객 출생년도,고객 성별,고객 등록일자,고객 등급,3개월 이내 금융/공동인증서 발급 여부,3개월 이내 사설인증서 발급 여부,3개월 이내 보안카드 및 OTP 발급 여부,3개월 이내 개인정보 수정 여부,탈옥 및 루팅 여부,...,C_Keylogging_Risk,D_Scripted_Like_Transaction,E_Vishing_Withdrawal_Risk,G_MoneyLaundering_Risk,F_MoneyLaundering_Risk,J_Suspicious_Small_Repeated_Transfer,K_Affiliate_Scam_Risk,거래금액,잔액 변화 비율,잔액이 크게 감소
0,0,1960,female,2003-01-07 10:59:08,E,1,0,0,0,1,...,0,1,0,0,0,1,0,-1645235,-0.144259,0
1,1,1960,female,2003-01-07 10:59:08,E,1,1,1,1,0,...,0,0,0,0,1,0,0,-2250000,-0.108097,0
2,2,1951,male,2003-01-06 18:10:55,B,1,1,1,1,0,...,0,0,0,1,0,0,1,3120000,0.150829,0
3,3,1999,female,2003-01-08 05:28:53,B,0,1,1,1,0,...,0,1,0,0,0,1,0,-16100000,-1.261722,0
4,4,1996,female,2003-01-17 03:37:22,A,0,1,0,0,1,...,0,1,0,0,0,1,0,515915,0.349939,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,119995,2004,male,2024-12-24 02:29:11,D,1,0,0,0,0,...,0,1,0,0,1,0,0,5163719,0.502680,0
119996,119996,1955,male,2010-07-15 01:27:01,B,0,0,1,0,0,...,0,1,0,0,1,0,0,-13946942,-1.020432,0
119997,119997,1987,female,2011-09-30 02:32:19,B,0,1,1,1,1,...,0,1,1,1,0,0,0,53460000,0.787947,0
119998,119998,2004,female,2024-04-02 11:15:06,C,1,1,1,1,0,...,0,1,0,0,0,1,0,2020000,0.176918,0


In [8]:


# ✅ 인코딩 및 컬럼 정제
categorical_cols = train_all.select_dtypes(include='object').columns
categorical_cols = [col for col in categorical_cols if col != 'label']
for col in categorical_cols:
    le = LabelEncoder()
    all_vals = pd.concat([train_all[col], test_all[col]], axis=0).astype(str)
    le.fit(all_vals)
    train_all[col] = le.transform(train_all[col].astype(str))
    test_all[col] = le.transform(test_all[col].astype(str))

train_all.columns = train_all.columns.str.replace(r"[^\w]", "_", regex=True)
test_all.columns = test_all.columns.str.replace(r"[^\w]", "_", regex=True)

feature_cols = [col for col in train_all.columns if col != 'label']
fraud_labels = [chr(ord('a') + i) for i in range(12)]  # 'a' ~ 'l'

# ✅ 모델 학습 함수 ################################### m 사이즈는 늘리면서 확인해보기
def train_optimal_model(train_df, target_label, feature_cols, n_models=20, m_size=2000):
    positive_df = train_df[train_df['label'] == target_label]

    # b~l: 100개씩
    other_frauds = train_df[(train_df['label'] != target_label) & (train_df['label'] != 'm')]
    fraud_neg = pd.concat([
        other_frauds[other_frauds['label'] == label].sample(n=100, random_state=seed)
        for seed, label in enumerate(other_frauds['label'].unique())
    ])

    # m: 2000개
    m_df = train_df[train_df['label'] == 'm'].sample(n=m_size, random_state=42)

    models = []
    for seed in range(n_models):
        binary_df = pd.concat([positive_df, fraud_neg, m_df])
        binary_df['target'] = (binary_df['label'] == target_label).astype(int)

        X = binary_df[feature_cols]
        y = binary_df['target']

        model = LGBMClassifier(class_weight='balanced', random_state=seed)
        model.fit(X, y)
        models.append(model)
    return models

# ✅ soft voting 예측
def predict_ensemble_avg(models, test_df, feature_cols):
    return np.mean([model.predict_proba(test_df[feature_cols])[:, 1] for model in models], axis=0)

# ✅ 전체 모델 학습
full_models = {}
for label in fraud_labels:
    print(f"▶ {label} 모델 학습 중...")
    full_models[label] = train_optimal_model(train_all, label, feature_cols)

# ✅ 테스트셋 예측
preds = pd.DataFrame()
for label in fraud_labels:
    preds[label] = predict_ensemble_avg(full_models[label], test_all, feature_cols)

# ✅ 최종 예측
threshold = 0.75 ########################## 0.75이하 의 값중에 최적값 찾기
final_pred = preds.idxmax(axis=1)
final_pred_with_m = final_pred.where(preds.max(axis=1) >= threshold, 'm')

# ✅ 저장
submission = pd.DataFrame({
    "ID": x_id,
    "Fraud_Type": final_pred_with_m
})
submission.to_csv("clf_submission.csv", index=False, encoding="utf-8-sig")

▶ a 모델 학습 중...
[LightGBM] [Info] Number of positive: 100, number of negative: 5100
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001218 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5085
[LightGBM] [Info] Number of data points in the train set: 5200, number of used features: 69
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p

[LightGBM] [Info] Number of positive: 100, number of negative: 5100
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001078 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5085
[LightGBM] [Info] Number of data points in the train set: 5200, number of used features: 69
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

[LightGBM] [Info] Number of positive: 100, number of negative: 5100
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001052 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5085
[LightGBM] [Info] Number of data points in the train set: 5200, number of used features: 69
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

[LightGBM] [Info] Number of positive: 100, number of negative: 5100
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001065 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5085
[LightGBM] [Info] Number of data points in the train set: 5200, number of used features: 69
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

[LightGBM] [Info] Number of positive: 100, number of negative: 5100
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001066 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5085
[LightGBM] [Info] Number of data points in the train set: 5200, number of used features: 69
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

...

[LightGBM] [Info] Number of positive: 100, number of negative: 5100
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001240 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5085
[LightGBM] [Info] Number of data points in the train set: 5200, number of used features: 69
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

[LightGBM] [Info] Number of positive: 100, number of negative: 5100
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001002 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5085
[LightGBM] [Info] Number of data points in the train set: 5200, number of used features: 69
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

[LightGBM] [Info] Number of positive: 100, number of negative: 5100
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001109 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5085
[LightGBM] [Info] Number of data points in the train set: 5200, number of used features: 69
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

[LightGBM] [Info] Number of positive: 100, number of negative: 5100
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001304 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5085
[LightGBM] [Info] Number of data points in the train set: 5200, number of used features: 69
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Info] Number of positive: 100, number of negative: 5100
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001076 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5085
[LightGBM] [Info] Number of data points in the train set: 5200, number of used features: 69
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Info] Nu

[LightGBM] [Info] Number of positive: 100, number of negative: 5100
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000941 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5085
[LightGBM] [Info] Number of data points in the train set: 5200, number of used features: 69
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Info] Number of positive: 100, number of negative: 5100
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001052 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5085
[LightGBM] [Info] Number of data points in the train set: 5200, number of used features: 69
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


In [9]:
print(submission['Fraud_Type'].value_counts())

Fraud_Type
m    80329
f     4987
b     4754
a     4747
k     4521
i     4438
e     4177
h     3355
j     2968
l     2574
c     1162
g     1098
d      890
Name: count, dtype: int64


In [10]:
print(submission['Fraud_Type'].value_counts())

Fraud_Type
m    80329
f     4987
b     4754
a     4747
k     4521
i     4438
e     4177
h     3355
j     2968
l     2574
c     1162
g     1098
d      890
Name: count, dtype: int64


In [11]:
# 클래스별 예측 결과 개수 출력
print(submission)

                 ID Fraud_Type
0       TEST_000000          j
1       TEST_000001          m
2       TEST_000002          m
3       TEST_000003          m
4       TEST_000004          h
...             ...        ...
119995  TEST_119995          m
119996  TEST_119996          m
119997  TEST_119997          m
119998  TEST_119998          m
119999  TEST_119999          m

[120000 rows x 2 columns]


In [12]:
fraud_labels = [chr(ord('a') + i) for i in range(12)]
fraud_labels

['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l']